# Small-model clause extraction test
Runtime -> Change runtime type -> **T4 GPU**. Tests whether a small model can turn ONE report sentence into `{concept, assertion}`.
Numbers are NOT extracted by the model (regex owns those).

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes

In [ ]:
# Add/remove any Hugging Face model id here. 4-bit loading keeps 3-4B models under ~3GB VRAM.
MODELS = [
    "google/gemma-3-1b-it",          # gated: accept license on HF first
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
    # "microsoft/Phi-3.5-mini-instruct",
    # "BioMistral/BioMistral-7B",    # ~5GB in 4-bit, will not fit a 4GB card
]

# Needed only for gated models (Gemma, Llama). Paste a read token when prompted.
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# (sentence, [(keyword expected in concept, expected assertion), ...])  -- all from your real sample reports
TESTS = [
 ("No focal lesion is seen.", [("focal lesion","ABSENT")]),
 ("No disc bulge is noted.", [("disc bulge","ABSENT")]),
 ("MPD is not dilated.", [("mpd","ABSENT")]),
 ("No e/o any calculus / mass seen in its lumen.", [("calculus","ABSENT"),("mass","ABSENT")]),
 ("No subluxation / dislocation / spondylolisthesis is seen.", [("subluxation","ABSENT"),("dislocation","ABSENT"),("spondylolisthesis","ABSENT")]),
 ("Mild acne on face, no hirsutism.", [("acne","PRESENT"),("hirsutism","ABSENT")]),
 ("Large right sided pneumothorax is seen with mild pleural effusion.", [("pneumothorax","PRESENT"),("effusion","PRESENT")]),
 ("Lung feilds are clear.", [("lung","NORMAL")]),
 ("Hila appear normal", [("hila","NORMAL")]),
 ("The facet joints are unremarkable.", [("facet","NORMAL")]),
 ("Cardio-thoracic ratio is within normal limits.", [("cardio","NORMAL")]),
 ("Loss of lumbar lordosis.", [("lordosis","PRESENT")]),
 ("Liver is enlarged in size (~15.8cm), normal in outline.", [("liver","PRESENT")]),
 ("IVD Spaces are normal", [("ivd","NORMAL")]),
 ("Grade 1 left ventricular diastolic dysfunction.", [("diastolic","PRESENT")]),
 ("Two small echogenic calculi measuring ~4mm are seen in lower calyx.", [("calcul","PRESENT")]),
]

In [ ]:
import json, re, time, gc, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

PROMPT = """Extract clinical facts from ONE sentence of a medical report.
Return ONLY JSON: {"items":[{"concept":"...","assertion":"PRESENT|ABSENT|NORMAL"}]}
Rules:
- Use only words that appear in the sentence. One item per finding or structure.
- no / not / without / no evidence of => ABSENT. normal / clear / unremarkable / within normal limits => NORMAL. Otherwise PRESENT.
- Do not output numbers or measurements. If nothing clinical, return {"items":[]}.

Sentence: No pericardial effusion is seen.
{"items":[{"concept":"pericardial effusion","assertion":"ABSENT"}]}
Sentence: Spleen is normal in size and Small calculus in the gallbladder.
{"items":[{"concept":"spleen","assertion":"NORMAL"},{"concept":"calculus in gallbladder","assertion":"PRESENT"}]}

Sentence: """

def load(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto", quantization_config=q)
    return tok, model

def ask(tok, model, sentence):
    user = PROMPT + sentence + "\n"
    try:
        text = tok.apply_chat_template([{"role":"user","content":user}], tokenize=False, add_generation_prompt=True)
    except Exception:
        text = user
    ids = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=120, do_sample=False, repetition_penalty=1.05, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

def parse(raw):
    i = raw.find("{")
    if i < 0: return None
    try:
        obj, _ = json.JSONDecoder().raw_decode(raw[i:])
        items = obj.get("items") if isinstance(obj, dict) else None
        if isinstance(items, list) and all(isinstance(x, dict) for x in items): return items
    except Exception:
        pass
    return None

def words(s): return set(re.findall(r"[a-z0-9]+", s.lower()))

def score(items, gold, sentence):
    found = correct = 0
    for kw, a in gold:
        m = [x for x in items if kw in str(x.get("concept","")).lower()]
        if m:
            found += 1
            correct += any(str(x.get("assertion","")).upper() == a for x in m)
    sw = words(sentence)
    ungrounded = 0
    for x in items:
        cw = words(str(x.get("concept","")))
        if cw and len(cw & sw) / len(cw) < 0.5: ungrounded += 1
    return found, correct, ungrounded

In [ ]:
summary, details = [], {}
for mid in MODELS:
    print("Loading", mid)
    try:
        tok, model = load(mid)
    except Exception as e:
        print("  SKIPPED:", str(e)[:150]); continue
    n_gold = sum(len(g) for _, g in TESTS)
    ok = found = correct = ungr = 0; t0 = time.time(); rows = []
    for sent, gold in TESTS:
        raw = ask(tok, model, sent); items = parse(raw)
        if items is not None:
            ok += 1; f, c, u = score(items, gold, sent); found += f; correct += c; ungr += u
        rows.append((sent, gold, items, raw))
    details[mid] = rows
    summary.append({"model": mid,
        "json_valid_%": round(100*ok/len(TESTS)),
        "concept_recall_%": round(100*found/n_gold),
        "assertion_correct_%": round(100*correct/n_gold),
        "ungrounded_items": ungr,
        "sec_per_sentence": round((time.time()-t0)/len(TESTS), 2),
        "vram_GB": round(torch.cuda.max_memory_allocated()/1e9, 2)})
    del model, tok; gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
pd.DataFrame(summary)

In [ ]:
# Inspect mistakes for one model
MODEL_TO_INSPECT = MODELS[0]
for sent, gold, items, raw in details.get(MODEL_TO_INSPECT, []):
    if items is None:
        print("BAD JSON |", sent, "\n   raw:", raw[:150].replace("\n"," ")); continue
    for kw, a in gold:
        m = [x for x in items if kw in str(x.get("concept","")).lower()]
        if not m or not any(str(x.get("assertion","")).upper() == a for x in m):
            print("WRONG    |", sent, "| expected", (kw, a), "| got", items)